In [ ]:
/*
================================================================================
  HANDLING HOT KEYS — EXPLAINED LIKE WE'RE JUST TALKING ABOUT IT
================================================================================

  Picture this: you run a big system with 100 identical cache servers,
  perfectly load-balanced. Everything's spread out evenly... until one
  single post goes viral. Suddenly EVERYONE on the internet wants that
  ONE specific piece of data — that ONE post, ONE product, ONE ticket
  listing.

  Here's the twist that trips people up: it doesn't matter that you
  have 100 servers. That ONE hot item still lives on exactly ONE of
  them. So while 99 servers are basically napping, that ONE server is
  getting absolutely hammered.

  THIS is the "hot key" problem: traffic piling up on ONE specific
  piece of data so hard that adding MORE generic servers doesn't help
  at all — because those new servers don't have a copy of the thing
  everyone actually wants.

  ANALOGY: Imagine a huge grocery store with 20 checkout lanes, all
  well staffed and running smoothly. Then a famous chef announces
  there's a secret discount code, but it only works at register #7.
  Now everyone rushes to register #7 specifically. It doesn't matter
  that lanes 1-6 and 8-20 are wide open — that one specific register is
  the bottleneck, and opening MORE random lanes elsewhere does nothing
  to fix it.


--------------------------------------------------------------------------------
  1. WHAT ACTUALLY MAKES A KEY "HOT"?
--------------------------------------------------------------------------------

  In a normal distributed system, data gets spread across many
  machines using something like a hash function — kind of like sorting
  mail into P.O. boxes based on the last digit of an address. Normally
  this spreads load evenly, and you scale by just adding more boxes
  (servers).

  A "hot key" breaks that assumption completely: one specific box gets
  WAY more mail than all the others combined, no matter how many other
  boxes you add.

  NORMAL DISTRIBUTION (nice and even):

        Node 1: 8K req/s    Node 2: 9K req/s    Node 3: 7K req/s
        Node 4: 8K req/s    Node 5: 8K req/s
        (total capacity: 50K req/s, well within limits)

  WITH A HOT KEY (all piled on ONE node):

        Node 1: 2K req/s    Node 2: 50,000 req/s !!  Node 3: 2K req/s
        Node 4: 2K req/s    Node 5: 2K req/s
        (Node 2 is now handling 5x its normal share, alone)

  A post goes viral, and EVERY request for "post:12345" hashes to the
  SAME node every single time — because that's literally what hashing
  does. Adding 50 more random nodes to the cluster does absolutely
  NOTHING for this specific problem, because none of those new nodes
  have a copy of post:12345 unless you specifically make them.

  IMPORTANT DISTINCTION — "hot SHARD" vs "hot KEY":

    A hot SHARD holds MANY different keys and happens to get more
    traffic than others — this you CAN usually fix by rebalancing or
    splitting that shard into smaller pieces.

    A hot KEY is literally ONE specific key. No amount of "rebalancing"
    helps, because every request for that EXACT key will always hash
    to the same place. You have to change HOW that one key is served
    (copy it, split it, cache it, or shield it) — not WHERE keys live.


--------------------------------------------------------------------------------
  2. QUESTIONS TO ASK BEFORE PICKING A FIX
--------------------------------------------------------------------------------

  Don't jump straight to "just add a cache!" Different kinds of hot
  keys need genuinely different solutions. Ask:

    Question                              | Why it matters
    -----------------------------------------|------------------------------------
    Is it being READ a lot, WRITTEN a lot,   | Read-hot keys can often just be
      or both?                              |  copied/cached. Write-hot keys need
                                             |  splitting or careful serialization
    Can the value be slightly STALE or       | Determines if a simple local cache
      approximate?                          |  or approximate counter is safe
    Is this protecting something SCARCE      | Inventory/seats/money usually need a
      (inventory, seats, money)?             |  single source of truth, not an
                                             |  approximate shortcut
    Does ORDER matter?                       | Queues/streams may need to stay in
                                             |  a single lane
    Is the spike PREDICTABLE (scheduled)      | Predictable = you can prepare ahead
      or a surprise (went viral)?            |  of time. Surprise = need automatic
                                             |  detection
    What's the fallback under overload?       | Serve stale data? Drop low-priority
                                             |  requests? Show a controlled error?

  Getting this wrong is a classic interview trap: treating a "viral
  post" (fine to be a little stale) the SAME way you'd treat "concert
  ticket inventory" (absolutely CANNOT be wrong, or you oversell seats).


--------------------------------------------------------------------------------
  3. WHY HOT KEYS ARE DANGEROUS (they don't just slow down — they CASCADE)
--------------------------------------------------------------------------------

  ANALOGY: Think of a single overloaded highway off-ramp during rush
  hour. First it just gets slow. Then people start impatiently trying
  alternate routes, which clogs THOSE roads too. Eventually there's a
  fender-bender because everyone's frustrated and cutting each other
  off, and now the ENTIRE area is gridlocked — not just that one ramp.

  Here's how a hot key cascades into a bigger outage, step by step:

        Hot Key Overloads Node
                │
                ▼
        Timeouts start accumulating
                │
                ▼
        Clients think it FAILED, so they RETRY (adding MORE load!)
                │
                ▼
        Node runs out of memory/connections → CRASHES
                │
                ▼
        Traffic fails over to ANOTHER node
                │
                ▼
        That node is now ALSO overloaded (same hot key, new victim)
                │
                ▼
        (cycle repeats...)

    STAGE 1 — Overload: the hot key exceeds the node's real capacity.
    Response times start climbing.

    STAGE 2 — Timeouts: clients wait too long, assume it failed, and
    RETRY — which piles even MORE requests onto the node that's already
    drowning.

    STAGE 3 — Collapse: the node runs out of memory or connections and
    crashes. Traffic fails over somewhere else.

    STAGE 4 — Propagation: the NEW node now gets hit with the hot-key
    traffic PLUS its own regular traffic, and quickly gets overloaded too.

  THE SNEAKY PART: your dashboards might show "average CPU across the
  cluster: totally fine, 15%!" — while ONE specific node is completely
  on fire. Averages hide hot keys. You need to look at the SPREAD, not
  just the average.


--------------------------------------------------------------------------------
  4. WHERE HOT KEYS SHOW UP (it's not just caches!)
--------------------------------------------------------------------------------

  Hot keys can appear at ANY layer where data gets split up by key:

    Layer                | Example hot key             | What happens
    ------------------------|--------------------------------|----------------------------------
    CDN                     | A viral video, breaking news    | Edge cache overwhelmed, origin
                            |  image                          |  server gets slammed too
    Cache (Redis, etc)      | Popular post, flash-sale item   | Single cache node saturated
    Database                | Popular user profile, trending  | Single database shard overloaded
                            |  topic                          |
    Message Queue           | Extremely high-volume channel   | One partition backs up badly
    Rate Limiter             | A global request counter        | The counter itself becomes the
                            |                                  |  bottleneck

  The cache layer is usually where you NOTICE it first, because caches
  sit directly in the path of tons of reads. But the underlying problem
  — "one key, way too much traffic" — is exactly the same at every layer.


--------------------------------------------------------------------------------
  5. PREDICTABLE vs UNPREDICTABLE HOT KEYS
--------------------------------------------------------------------------------

  ANALOGY: There's a big difference between preparing for a scheduled
  Black Friday sale (you KNOW it's coming, you can staff up in
  advance) versus a random product suddenly going viral on TikTok
  overnight with zero warning.

    PREDICTABLE (you can prepare ahead of time):
      - Flash sales
      - Scheduled product launches
      - Ticket releases for a known event
      - A celebrity's scheduled post

    UNPREDICTABLE (no warning, needs automatic response):
      - Random viral content
      - Breaking news
      - An unexpected surge in reshares
      - A surprise mention from someone huge

  For PREDICTABLE hot keys, you get to be proactive — pre-warm caches,
  set up dedicated resources, have your team on standby. For
  UNPREDICTABLE ones, you need systems that DETECT and ADAPT
  automatically, in real time, because nobody saw it coming.


--------------------------------------------------------------------------------
  6. DETECTING HOT KEYS (you can't fix what you can't see)
--------------------------------------------------------------------------------

  6.1 PROACTIVE DETECTION — watch continuously, catch it early

    Some cache systems (like Redis) let you sample which keys are
    getting accessed most. Look for the SAME key showing up over and
    over on an overloaded node — that's your smoking gun.

    Built-in tools (like Redis's hot-key sampling feature) can help,
    but they're sampling-based, so they won't catch EVERYTHING. In
    practice, teams combine several signals: cache stats, request
    logs, and alerts specifically about node IMBALANCE.

  6.2 REACTIVE DETECTION — your safety net when something slips through

    Watch for signs of IMBALANCE across your nodes:

    Metric              | Normal pattern              | Hot-key pattern
    ----------------------|--------------------------------|--------------------------------
    Node CPU               | All nodes similar (within 20%) | ONE node way higher than rest
    Request latency          | Roughly uniform across nodes  | ONE node has a huge latency spike
    Network I/O              | Evenly spread out              | ONE node completely saturated
    Cache evictions           | Low everywhere                | High on just ONE node

    A GOOD ALERT RULE: "fire an alert if any single node's CPU is above
    70% WHILE the cluster's average CPU is still under 30%." That
    combination specifically catches SKEW (imbalance), not just
    overall high load — which is exactly what a hot key looks like.

    Important: an alert firing should trigger INVESTIGATION, not
    blind automatic action. Sometimes imbalance has other causes
    entirely — uneven connections, bad routing, a noisy background job.

  6.3 PREDICTIVE DETECTION — for stuff you already know is coming

    For scheduled events (flash sales, launches), you don't need to
    "detect" anything — you already KNOW what's about to get hot.

    PRE-WARMING STRATEGY (4 steps):
      1. Figure out WHICH keys will be hot (product details, inventory
         count, pricing — for a product launch, say)
      2. Pre-copy those read-heavy keys onto MULTIPLE cache nodes ahead
         of time, and pre-load ("warm") them with real data
      3. Set up special handling per data type — route reads across the
         replicas, group concurrent requests together, use a proper
         reservation system for anything scarce like inventory
      4. Have an operations plan ready: who to call, pre-approved ways
         to scale up capacity, and a rollback plan, just in case

    For predictable events, this kind of prep is often the ENTIRE
    difference between "smooth sailing" and "3am incident call."


--------------------------------------------------------------------------------
  7. THE SOLUTION TOOLKIT — 7 WAYS TO HANDLE A HOT KEY
--------------------------------------------------------------------------------

  There's no single fix-all. Different situations call for different
  tools. Let's go through them.

  7.1 LOCAL CACHING (cache it right inside each app server's memory)

    ANALOGY: Instead of every employee walking to the one shared filing
    cabinet in the basement every single time they need a common form,
    everyone just keeps a personal photocopy of that form at their own
    desk. Way fewer trips to the basement.

        Incoming Requests
              │
        ┌─────┼──────────────┬─────────────┐
        ▼     ▼               ▼
      App1   App2    ...    AppN
      (local (local          (local
       cache) cache)          cache)
        │       │              │
        └───────┴──────┬───────┘
                        ▼ (only ~5% of requests, on a local cache MISS)
                     Redis

    If you have 100 app servers, each keeping a small local cache of
    hot data, you've spread the read load across 100 separate
    processes instead of hammering ONE shared Redis node. Even a tiny
    5-second local cache can massively cut down on Redis traffic for
    read-heavy data.

    THE COST: staleness. With a 5-second local cache, data could be up
    to 5 seconds old. Totally fine for a "like count." NOT fine for a
    flash-sale inventory count where an outdated number could mean you
    oversell.

    Pros                              | Cons
    ------------------------------------|------------------------------------
    Cuts down on distributed cache calls | Data can be slightly stale
    Very low latency on hits              | Uses memory on every app server
    Usually no infrastructure changes     | Different servers might briefly
                                          |  disagree on the "current" value
    Read capacity grows with app servers  | Needs careful TTL tuning

    BEST FOR: viral posts, product descriptions, public profiles —
    anything read-heavy where a few seconds of staleness is a non-issue.

  7.2 KEY REPLICATION (copy the hot key onto multiple cache nodes)

    ANALOGY: Instead of ONE librarian holding the only copy of this
    week's bestseller, the library photocopies it and puts 5 copies on
    5 different shelves. Now 5 people can grab a copy simultaneously
    instead of queuing for the one original.

        Request for hot key
              │
              ▼
        Load Balancer (round-robin)
              │
      ┌───────┼───────┐
      ▼       ▼        ▼
    Node 1  Node 2   Node 3
   (copy of (copy of (copy of
   the key)  the key)  the key)

    Instead of one physical key, you store the SAME data under several
    key names (like post:12345:r1, post:12345:r2...), each of which
    naturally hashes to a DIFFERENT node. With 5 replicas, you can
    spread read traffic across 5 nodes instead of 1 — nearly 5x the
    read capacity if things stay balanced.

    IMPORTANT: don't assume your chosen replica is always healthy — if
    one replica is down, fall back to ANOTHER replica (not straight to
    the origin database).

    THE COST: every UPDATE now has to touch or invalidate ALL the
    replicas, and for a brief window, different users might see
    slightly DIFFERENT versions of the data unless you add versioning
    or short TTLs.

    Pros                          | Cons
    ---------------------------------|-------------------------------------
    Spreads read load across nodes    | Extra storage (multiple copies)
    Doesn't need per-process caching   | Must update ALL replicas on write
    Meaningfully adds read capacity     | Some write amplification
    Simple to route reads               | Possible brief inconsistency

    BEST FOR: high read-to-write ratio data where some bounded
    staleness is fine — public profiles, product details, config data.

  7.3 KEY SPLITTING / SHARDED COUNTERS (for WRITE-heavy hot keys)

    ANALOGY: Instead of ONE cash register trying to ring up every
    single customer in a massive store (impossible), you open 100
    smaller registers, and at the END of the day, someone adds up all
    100 registers' totals to get the grand total.

    This is for a DIFFERENT problem than sections 7.1/7.2 — those were
    about READ-heavy hot keys. This is for WRITE-heavy ones, like a
    counter getting incremented like crazy (think: "likes" on a viral
    post).

    Copying the SAME counter to multiple places (like section 7.2)
    doesn't help here, because every single write still has to update
    the SAME logical number. Instead, you SPLIT the counter itself:

        BEFORE:  likes:post:123  →  50,000 writes/sec (one key, overloaded)

        AFTER (split into 100 shards):
          likes:post:123:0, likes:post:123:1, ... likes:post:123:99
          (each shard absorbs roughly 500 writes/sec instead of 50,000)

    THE COST MOVES TO THE READ SIDE: to get the TOTAL like count, you
    now have to read ALL 100 shards and add them up. In practice you'd
    fetch them all in one batched request (not 100 separate round
    trips), and cache the ADDED-UP total for a short time rather than
    recomputing it on every single page view.

    Pros                              | Cons
    ------------------------------------|-------------------------------------
    Spreads out write load                | Reading requires aggregating
    Capacity scales with shard count       | More complex client-side logic
    Great for counters/metrics             | NOT for exact atomic reads
    Removes the single-key bottleneck       | Can lose strict ordering

    BEST FOR: like counts, view counts, impression counts, approximate
    rate tracking.

    ⚠️ NOT BEST FOR: scarce inventory, bank balances, seat reservations.
    Those NEED a single, exact source of truth — you can't approximate
    "how many concert seats are left," or you risk overselling.

    A MIDDLE-GROUND TRICK worth knowing: if a number must stay EXACTLY
    correct but you can't afford one database write per single event,
    have each server buffer up increments IN MEMORY and flush the
    SUM every few hundred milliseconds as one atomic update. This
    collapses thousands of tiny writes into a handful of bigger ones,
    while keeping the final total exact — at the cost of a small delay
    before a write is "durable," and the risk of losing that in-memory
    buffer if the server crashes before it flushes.

  7.4 REQUEST COALESCING ("Only one of you needs to go get it")

    ANALOGY: Three roommates all separately notice you're out of milk
    and each start driving to the store. Instead, imagine if the first
    one to notice just said "I got it, hang tight" — and the other two
    just... waited for that ONE trip's milk instead of also driving out.

    This specifically solves the "cache stampede" problem: when a
    cache key expires and MANY concurrent requests all try to
    re-fetch it from the database AT THE SAME TIME.

        Client 1 ─┐
        Client 2 ─┼──▶ App Server ──ONE fetch──▶ Redis/Database
        Client 3 ─┘         │
                             ▼
                  All 3 clients get served
                  from that ONE fetch's result

    Only the FIRST request actually goes and fetches the data.
    Everyone else who asks for the SAME key while that fetch is
    in-flight just... waits, and gets handed the SAME result once it's
    ready. This is sometimes called "singleflight."

    IMPORTANT GOTCHA: if that one fetch FAILS, you need to make sure
    everyone who was WAITING also gets told "it failed, try again" —
    otherwise they'll all hang forever waiting on a result that's
    never coming.

    THE LIMIT: this only helps requests hitting the SAME server at the
    SAME moment. It doesn't fix sustained hot-key load across your
    WHOLE cluster unless you combine it with something more (like a
    distributed lock, described in 7.6).

    Pros                              | Cons
    ------------------------------------|-------------------------------------
    Kills the "thundering herd" on misses | Only helps CONCURRENT requests
    No extra persistent storage needed     | Only within a single server (unless
                                          |  you add cross-server coordination)
    Great fit for cache-miss bursts         | Doesn't fix SUSTAINED heavy load

    BEST FOR: cache stampedes, cold cache warm-up, bursty traffic.

  7.5 STALE-WHILE-REVALIDATE ("Serve the old one while I grab a fresh one")

    ANALOGY: A newsstand doesn't take down yesterday's newspaper the
    INSTANT midnight hits and leave an empty rack until the new one
    arrives. They keep yesterday's paper out for a bit while today's
    is being printed, so the rack is never empty.

    When a cache entry expires, instead of making EVERY reader wait
    for a fresh fetch, you keep serving the (slightly old) cached value
    immediately, WHILE quietly kicking off ONE background refresh:

        Read Request
              │
              ▼
        Is the cache entry Fresh, Stale-but-in-grace-window, or
        Expired-past-grace-window?
              │
      ┌───────┼─────────────────┐
      ▼       ▼                  ▼
    Fresh?  Stale but OK?    Expired too long?
      │       │                  │
      ▼       ▼                  ▼
    Return  Return the OLD    Fetch fresh data
    value   value AND trigger  SYNCHRONOUSLY
            ONE background     (reader has to wait
            refresh (reader    this one time)
            doesn't wait)

    THE COST: readers might briefly see data that's one refresh-cycle
    old. And the VERY FIRST request after the "grace window" runs out
    still has to wait for a real fetch. Pairing this with request
    coalescing (7.4) keeps even THAT down to just one origin call.

    Pros                                | Cons
    --------------------------------------|-----------------------------------
    Reads stay FAST even after expiry      | Readers can see slightly old data
    Origin gets ONE refresh, not a stampede | The very first post-grace request
                                          |  still has to wait
    Easy to add on top of existing TTL      | Needs a defined "how stale is OK"
    caches                                |  window per key

    BEST FOR: public content, product pages, profiles, live metadata —
    basically anywhere "a little bit old" beats "totally down."

  7.6 READ-THROUGH CACHE WITH LOCKING (coordinate ACROSS servers)

    This builds on request coalescing (7.4), but solves the "across
    multiple SERVERS" version of the same problem, using a distributed
    lock (something like Redis) that all servers can see.

    ANALOGY: Multiple different STORE LOCATIONS (not just employees at
    one store) all notice they're out of a popular item at the same
    time. Instead of each location separately calling the warehouse,
    they use a shared sign-up sheet: whoever signs up FIRST calls the
    warehouse, and the other locations just wait for that one call's
    results to get shared with them.

        Server 1                Server 2            Cache      Database
           │                       │                   │           │
           │──GET key (miss)──────▶│                   │           │
           │                       │──GET key (miss)──▶│           │
           │──ACQUIRE lock:key────────────────────────▶│           │
           │◀────lock acquired!────────────────────────│           │
           │                       │──ACQUIRE lock:key─▶│           │
           │                       │◀──lock held, wait──│           │
           │──query the database────────────────────────────────▶│
           │◀───────────────────data─────────────────────────────│
           │──SET key {data}, RELEASE lock──────────────▶│         │
           │                       │──GET key (now a HIT!)────────▶│
           │                       │◀───────{data}──────│          │

    ONLY the server that grabs the lock actually queries the database.
    Everyone else waits (or serves stale data) until the lock is
    released and the fresh value is in the cache.

    IMPORTANT SAFETY DETAIL: releasing a lock needs to check "is this
    lock even still MINE?" before deleting it — otherwise a server
    whose lock already expired could accidentally delete a DIFFERENT
    server's currently-active lock. This is usually done as a single
    small atomic script, not two separate steps.

    Pros                              | Cons
    ------------------------------------|-------------------------------------
    Coordinates across MULTIPLE servers   | Waiting servers experience delay
    Protects the database from stampedes  | Lock management adds real complexity
    Plays nicely with TTL-based caching     | Buggy lock handling = stuck systems

    BEST FOR: expensive database queries, cache refresh storms,
    coordinated cache warming across a whole fleet of servers.

  7.7 RATE LIMITING PER KEY ("limit the damage, don't fix the cause")

    ANALOGY: A bouncer at an overcrowded club doesn't try to make the
    club magically bigger — they just stop letting more people in past
    a certain point, to protect everyone already inside.

        Requests
           │
           ▼
       Rate Limit Check
           │
      ┌────┴────┐
      ▼         ▼
   Under      Over the
   limit       limit
      │         │
      ▼         ▼
   Process   Serve stale data
             or return an error

    Sometimes the best move isn't a clever technical trick — it's just
    saying "no more" past a certain point, so the WHOLE system doesn't
    go down trying to serve every single request perfectly.

    Pros                          | Cons
    ---------------------------------|-------------------------------------
    Protects the whole system          | Some users see errors/stale data
    Gives you a clear overload boundary | Doesn't actually FIX the hot key
    Degrades gracefully instead of      | Needs careful per-key tuning
    crashing entirely                  |

    BEST FOR: your LAST line of defense, always combined with the
    other strategies above — not a replacement for them.


--------------------------------------------------------------------------------
  8. CHOOSING THE RIGHT FIX — A DECISION GUIDE
--------------------------------------------------------------------------------

  Two SEPARATE questions decide your approach:

  QUESTION 1 — "What TYPE of thing is hot?"

                        What is hot?
                             │
          ┌──────────────────┼───────────────────┐
          ▼                  ▼                    ▼
      Read-heavy        Counter/metric     Scarce or ordered
                                             state (inventory,
                                             seats, queues)
          │                  │                    │
          ▼                  ▼                    ▼
    Bounded staleness    Approximate OK?      Use a queue, single
    OK?                                        writer, or reservation
          │                  │                  flow — NEVER approximate
      ┌───┴───┐          ┌───┴───┐              this!
      ▼       ▼          ▼       ▼
     Yes      No         Yes      No
      │       │           │       │
      ▼       ▼           ▼       ▼
   Local    Versioned   Sharded  Split writes
   Cache /  replicas    Counter  + cache reads
   Stale-   or strong
   While-   source of
   Revali-  truth
   date

  QUESTION 2 — "What's the SHAPE of the traffic?" (decides the ROLLOUT)

    Traffic pattern?
          │
     ┌────┼──────────────┬───────────────┐
     ▼    ▼               ▼
   Bursty            Sustained        Predictable
     │                   │                 │
     ▼                   ▼                 ▼
   Request Coalescing  Replication +   Pre-warm +
   + Locking            Backpressure   special routing

  QUICK REFERENCE TABLE:

    Scenario                        | Primary fix                       | Secondary fix
    ------------------------------------|--------------------------------------|--------------------------
    Viral post (read, stale OK)          | Local cache / stale-while-revalidate  | Key replication
    Flash sale PRODUCT PAGE (read-heavy) | Pre-warm + replicated cache            | CDN / local cache
    Flash sale INVENTORY (scarce!)       | Queue/reservation or single writer     | Waiting room / backpressure
    Live score (read + write)            | Local cache + versioned updates         | Key replication
    Cache stampede (bursty)              | Request coalescing                       | Locking / stale-while-revalidate
    High-follower profile (read-heavy)   | Key replication                          | Local cache / CDN
    Rate limiter counter                 | Sharded/local counters                    | Centralized token service


--------------------------------------------------------------------------------
  9. IN PRACTICE — LAYER YOUR DEFENSES (defense in depth)
--------------------------------------------------------------------------------

  Real systems basically NEVER rely on just ONE of these techniques.
  You stack them, so each layer soaks up traffic before it reaches
  the next, more expensive layer:

    Request
       │
       ▼
    Layer 1: CDN                  → caches static assets/images/video
       │ (miss)
       ▼
    Layer 2: Local Cache          → app server memory, ~5 second TTL
       │ (miss)
       ▼
    Layer 3: Replicated Cache      → hot keys copied across multiple nodes
       │ (miss)
       ▼
    Layer 4: Database              → protected miss path, rarely reached
       │
       ▼
    Layer 5: Rate Limiting          → protects everything from total overload

  EXAMPLE: Handling a truly viral post, end to end:

    Layer               | Role                    | What it actually does
    -----------------------|----------------------------|-------------------------------------
    1. CDN                  | Absorb anonymous reads      | Caches images/videos/thumbnails
                            |                             | and API responses with a short TTL
    2. Local cache (app)    | Cut repeated cache calls     | Small in-memory LRU cache, a few
                            |                             | seconds TTL, accepts slight staleness
    3. Replicated cache     | Spread hot reads             | Hot posts copied across nodes,
                            |                             | plus request coalescing
    4. Database (rare)      | Protect the source of truth   | Dedicated read replica, locking to
                            |                             | prevent stampedes — reached RARELY
    5. Protection            | Limit overload impact         | Per-user limits, per-key budgets,
                            |                             | graceful fallback to stale data

  By the time a request actually reaches the DATABASE, it should
  ideally only be a genuine cache miss or a scheduled refresh — almost
  everything else got absorbed by an earlier, cheaper layer.


--------------------------------------------------------------------------------
  10. QUICK-REFERENCE DECISION CHECKLIST
--------------------------------------------------------------------------------

  Before finalizing your hot-key strategy, ask:

    [ ] Have I clearly identified WHAT is hot — read-heavy content, a
        write-heavy counter, or scarce/ordered state? (These need
        completely different solutions — don't mix them up!)
    [ ] Can this data tolerate being slightly STALE? If yes, local
        caching or stale-while-revalidate is probably my first move.
    [ ] Is this protecting something SCARCE (inventory, seats, money)?
        If yes, I should NOT use approximate counters or sharding —
        I need a single source of truth, queue, or reservation system.
    [ ] Do I have DETECTION in place — will I actually notice a hot
        key forming, or am I just hoping average metrics look fine?
    [ ] Is this spike PREDICTABLE (I can pre-warm) or a SURPRISE (I
        need automatic, real-time detection and response)?
    [ ] Am I layering multiple defenses together (CDN → local cache →
        replicated cache → database → rate limiting), instead of
        betting everything on just one trick?
    [ ] If I'm using locks or coalescing, have I handled the FAILURE
        path — do waiting requests eventually get told "it failed, try
        again" instead of hanging forever?
    [ ] What's my fallback when things get truly overloaded — serve
        stale data, queue people, or return a controlled error? Have I
        actually decided this on purpose, or will it just be undefined
        behavior during an actual incident?


--------------------------------------------------------------------------------
  TL;DR — THE ONE THING TO REMEMBER
--------------------------------------------------------------------------------

  A hot key breaks the assumption that "more servers = more capacity,"
  because one specific piece of data always lives in one specific
  place, no matter how many other servers you add. The fix isn't
  "add more capacity everywhere" — it's changing HOW that one specific
  key gets served: copy it (replication), split it (sharded counters),
  cache it locally, coordinate around it (coalescing/locking), or
  shield the rest of the system from it (rate limiting). Real systems
  stack several of these together as layered defense, because no
  single trick handles every kind of hot key on its own.

================================================================================
*/